# Thesis + IMU (`thesis_imu`) — Jacobi message passing with the gyro as a soft R anchor (walkthrough)

**What this unrolls**, in the order the harness runs it (`evaluation.py::experiment_tracking`, `thesis_imu` branch):

| step | code |
|---|---|
| run config, frame-0 R from gyro | `run_config.py::RunConfig` |
| event frames `V` | `interacting_maps/data_loader.py::EventFrameSequence` |
| kinematic matrix `C`, normal equations `M⁻¹` | `interacting_maps/camera.py::build_kinematic_matrix`, `build_R_normal_equations`, `solve_R_lstsq` |
| network build | `run_config.py::make_network` (thesis branch) → `InteractingMapsThesis.__init__`, `initialize_from_rotation` |
| **one iteration**, every update/message written out | `interacting_maps/network_dissertation.py::InteractingMapsThesis.step(V, omega_imu=ω_gyro)` → Phase 1 `Cost_OFCE`, `Cost_Spatial`, `Cost_Kinematics`, **`Cost_IMU`**; Phase 2 `Quantity.update` + clip |
| the `n_iters` loop on one frame | same, recorded per iteration |
| multi-frame loop + metrics | `evaluation.py::experiment_tracking`, `metrics.py::compute_metrics` |

Every unrolled block ends with an assertion against the real module function.
Inputs: `data/poster_rotation/` (seg_C — the same segment as `cmax_v1/v2_walkthrough.ipynb`, so the four notebooks
compare directly). Outputs: figures + one tracking run under `experiments/model_walkthrough/`.

**Thesis + IMU = the thesis network plus one more message on R.** Each frame `step` sets `Cost_IMU.R_imu = ω_gyro·dt`, and every iteration `R` receives `δ_IMU·(R − ω_gyro·dt)` next to `Cost_Kinematics`' `δ_FR·(R − R*)`. The gyro is a model **input**, so scoring is against Vicon (`groundtruth.txt`), never the gyro (CLAUDE.md, "two ground truths").

In [ ]:
# setup — run from the repo root (data/ and experiments/ are relative paths)
import os, sys
os.environ.setdefault('IM_MLFLOW', '0')          # walkthrough runs never go to MLflow
ROOT = os.path.abspath('.')
assert os.path.exists(os.path.join(ROOT, 'config.py')), 'start the notebook from the repo root'
sys.path.insert(0, ROOT)
# local slice switches (env): WT_NFRAMES (default 5), WT_NITERS (default ITERS_PER_FRAME), WT_K (frame studied in detail)

In [ ]:
# imports — only what the cells use, grouped by module
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from config import ITERS_PER_FRAME
from run_config import RunConfig, make_network
from interacting_maps.data_loader import EventFrameSequence, CameraCalibration
from interacting_maps.eval_io import (load_imu, load_groundtruth, load_omega_gt,
                                      get_gyro_for_frame, get_reference_omega)
from interacting_maps.metrics import compute_metrics, curl_share
from interacting_maps.network_dissertation import InteractingMapsThesis
from interacting_maps.camera import (build_kinematic_matrix, build_R_normal_equations, solve_R_lstsq,
                                     CLIP_I, CLIP_G, CLIP_F, CLIP_R)
from interacting_maps.style import use_style, save, to_deg_s, OMEGA_AXIS_COLORS, SCREEN_WIDTH
from interacting_maps.viz import draw_map, map_scale, net_maps
import evaluation as E   # only for the final end-to-end check (experiment_tracking)

In [ ]:
# constants — the walkthrough's knobs (same segment as the CMax walkthroughs: poster seg_C, ω_x-dominant, |ω|≈1.1 rad/s)
DATASET    = 'poster_rotation'
SEGMENT_ID = 'seg_C'                                         # config.DATASET_SEGMENTS
MODEL      = 'thesis_imu'
N_FRAMES   = int(os.environ.get('WT_NFRAMES', 5))            # segment default is 150
N_ITERS    = int(os.environ.get('WT_NITERS', ITERS_PER_FRAME))   # config.ITERS_PER_FRAME = 75
K          = int(os.environ.get('WT_K', 0))                  # frame studied step by step
OUT_ROOT   = 'experiments/model_walkthrough'                 # NOT results/ (load-bearing)
FIG_DIR    = os.path.join(OUT_ROOT, 'thesis_imu_figures')
os.makedirs(FIG_DIR, exist_ok=True)
np.set_printoptions(precision=4, suppress=True)
assert 0 <= K < N_FRAMES

## 1) Run configuration — `run_config.py::RunConfig`
`RunConfig(model='thesis_imu')` (the `else` branch) picks `THESIS_PARAMS` and sets `use_thesis=True`, `use_imu=True`; the folder name gains `_dimu{δ_IMU}`. `initial_R` is the gyro mean over the first frame window × `dt` (`RunConfig._compute_initial_R`), and the same gyro is the anchor target on every frame.

In [ ]:
# run_config.py::RunConfig.__init__
rc = RunConfig(dataset=DATASET, model=MODEL, segment=SEGMENT_ID,
               n_frames=N_FRAMES, n_iters=N_ITERS, out_root=OUT_ROOT)
rc

In [ ]:
# the switches this model sets, the deltas, and the frame-0 R (rad/frame and °/s)
print({k: getattr(rc, k) for k in ('use_thesis', 'use_imu', 'use_cmax', 'use_cmax_v2',
                                   'distortion_mode', 'poisson', 'frame_duration')})
print('params     :', rc.params)
print('initial_R  :', rc.initial_R, 'rad/frame  =', to_deg_s(rc.initial_R / rc.frame_duration), '°/s')
print('output_dir :', rc.output_dir)

## 2) Event frames `V` — `interacting_maps/data_loader.py::EventFrameSequence`
The network's only per-frame input. Events are read once for `[t_start, t_start + n·dt + 0.1 s)`; each frame bins its
window into a signed count image, clipped at ±10 and divided by 10 (`clip_value=10.0` in `experiment_tracking`).
In `C_full` mode the events are **not** undistorted here (lens handled inside the kinematic matrix `C`).

In [ ]:
# evaluation.py::experiment_tracking — EventFrameSequence(...) exactly as the harness builds it
seq = EventFrameSequence(
    rc.paths['events'], rc.paths['calib'],
    frame_duration=rc.frame_duration, t_start=rc.t_start,
    n_frames=rc.n_frames, clip_value=10.0,
    undistort=rc.undistort_at_event_level,
    sensor_size=rc.sensor_size,
)
H, W = seq.H, seq.W
fx, fy, cx, cy = seq.calib.fx, seq.calib.fy, seq.calib.cx, seq.calib.cy
frames = list(seq)                         # [(V, t_mid), ...]
print('H, W:', H, W, '| undistort at event level:', seq.undistort, '| dist coeffs:', seq.calib.dist)

In [ ]:
# what V looks like: per frame, share of pixels with events and of saturated pixels (|V| = 1 ↔ ≥10 net events)
pd.DataFrame([{'frame': k, 't_mid': t, 'nonzero_px': int((V != 0).sum()),
               'nonzero_share': (V != 0).mean(), 'saturated_px': int((np.abs(V) >= 1).sum()),
               'V_min': V.min(), 'V_max': V.max()}
              for k, (V, t) in enumerate(frames)])

In [ ]:
# sensors: Vicon (independent SCORING reference) and the gyro (frame-0 init; for thesis_imu also the anchor)
imu_data = load_imu(rc.paths['imu'])
gt_data  = load_groundtruth(rc.paths['groundtruth'])
om_data  = load_omega_gt(rc.paths.get('omega_gt'))
print('imu rows', len(imu_data), '| gt rows', len(gt_data), '| omega_gt', om_data is not None)

## 3) Kinematic matrix `C` — `interacting_maps/camera.py::build_kinematic_matrix`
Built once in `InteractingMapsThesis.__init__`. Per pixel a 2×3 matrix maps ω (rad/frame) to pixel flow `F = C·R` (px/frame). In `C_full`
every native (distorted) pixel is back-projected to its true undistorted normalized coords `(x', y')` with
`cv2.undistortPoints`, the rotational flow `A(x',y')` is evaluated there and mapped back into distorted-pixel space by
the Brown–Conrady Jacobian: `C = diag(fx,fy)·J_D·A`. This is the only place the lens enters the network.

In [ ]:
# run_config.py::make_network — dist_coeffs for C_full (None for undistort_events → pinhole C)
dist_coeffs = CameraCalibration(rc.paths['calib']).dist if rc.distortion_mode == 'C_full' else None
dist_coeffs

In [ ]:
# camera.py::build_kinematic_matrix — distortion-aware C, and the pinhole C for comparison
C_mat = build_kinematic_matrix(H, W, fx, fy, cx, cy, dist_coeffs=dist_coeffs, include_jacobian=True)
C_pin = build_kinematic_matrix(H, W, fx, fy, cx, cy, dist_coeffs=None)
print('C_mat', C_mat.shape)
for name, (r, c) in (('centre', (H // 2, W // 2)), ('corner', (0, 0))):
    print(f'{name} pixel (row {r}, col {c}):\n', C_mat[r, c], '\n  pinhole:\n', C_pin[r, c])

In [ ]:
# what the lens does to C: relative change of each pixel's C vs pinhole
rel = np.linalg.norm(C_mat - C_pin, axis=(2, 3)) / np.linalg.norm(C_pin, axis=(2, 3))
print(f'|C_full − C_pinhole| / |C_pinhole|: centre {rel[H // 2, W // 2]:.3%}, median {np.median(rel):.2%}, max {rel.max():.2%}')

### 3b) Normal equations — `camera.py::build_R_normal_equations`, `solve_R_lstsq`
`R* = argmin_R Σ‖F − C·R‖² = M⁻¹·ΣCᵀF` with the constant `M = ΣCᵀC` (3×3), precomputed once. Both networks move `R`
toward this `R*`. Its conditioning says how well the whole flow field pins down each axis: ω_x/ω_y produce
near-uniform translations of the image (huge weight in `M`), ω_z a rotation about the centre that is small near it.

In [ ]:
# camera.py::build_R_normal_equations — M = Σ CᵀC, inverted
M = np.einsum('hwji,hwjk->ik', C_mat, C_mat)
M_inv = np.linalg.inv(M)
print('M =\n', M, '\ncond(M) =', np.linalg.cond(M))
print('per-axis sensitivity sqrt(diag M) (px per rad/frame, summed over pixels):', np.sqrt(np.diag(M)))
assert np.allclose(M_inv, build_R_normal_equations(C_mat))

In [ ]:
# camera.py::solve_R_lstsq — R* from a flow field; on the exact field F = C·R it returns R (sanity of the solve)
F_init = np.einsum('hwij,j->hwi', C_mat, rc.initial_R)
R_star_init = M_inv @ np.einsum('hwji,hwj->i', C_mat, F_init)
assert np.allclose(R_star_init, solve_R_lstsq(M_inv, C_mat, F_init))
print('initial R  :', rc.initial_R, '\nR*(C·R)    :', R_star_init)
mag = np.hypot(F_init[..., 0], F_init[..., 1])
print(f'|F| of the initial flow C·R: mean {mag.mean():.3f} px/frame, max {mag.max():.3f}')

In [ ]:
# the initial flow field C·R_init (direction = colour, |F| = brightness)
with use_style('screen'):
    fig, ax = plt.subplots(1, 1, figsize=(SCREEN_WIDTH / 3, 2.4))
    draw_map(ax, 'F', F_init, title=r'$F = C\cdot R_{init}$')
    path = os.path.join(FIG_DIR, f'{MODEL}_F_init.png'); save(fig, path)
display(Image(path))

## 4) Network — `run_config.py::make_network` (thesis branch, unrolled)
`InteractingMapsThesis(...)` builds `C`, the quantities `V, I, G, F, R` (each with a value and a gradient accumulator)
and the cost list `[Cost_OFCE, Cost_Spatial, Cost_Kinematics, Cost_IMU]` (`Cost_Integrability`/`Cost_MapPrior` only if
their deltas are > 0). `Cost_IMU` is always *in* the list; `step` skips it unless `omega_imu` is given.
Then `initialize_from_rotation` sets `R = initial_R`, `F = C·R` (mutually consistent start — otherwise
`Cost_Kinematics` would pull R toward `R*(F=0) = 0` on the first iteration) and `I` = 0.01·N(0,1) with seed 42 on an
`H×W` grid; `G = 0`.

In [ ]:
# run_config.py::make_network — InteractingMapsThesis(...)
net = InteractingMapsThesis(H=H, W=W, fx=fx, fy=fy, cx=cx, cy=cy,
                            frame_duration=rc.frame_duration, dist_coeffs=dist_coeffs,
                            poisson=rc.poisson, **rc.params)
print('costs:', [type(c).__name__ for c in net.costs], '| r_update:', net.r_update,
      '| kinematics updates R:', net.cost_kin.update_r)
assert np.array_equal(net._C_mat, C_mat) and np.allclose(net.cost_kin._M_inv, M_inv)

In [ ]:
# network_dissertation.py::initialize_from_rotation — R = initial_R, F = C·R, I = 0.01·N(0,1) (seed 42), G = 0
net.q_R.value = rc.initial_R.copy()
net.q_F.value = np.einsum('hwij,j->hwi', net._C_mat, rc.initial_R)
net.q_I.value = np.random.default_rng(42).standard_normal((H, W)) * 0.01
print('I', net.I.shape, 'std', net.I.std(), '| G max', np.abs(net.G).max(), '| R', net.R)

In [ ]:
# check: make_network == unrolled build
net_chk = make_network(rc, H, W, fx, fy, cx, cy)
for q in ('q_I', 'q_G', 'q_F', 'q_R'):
    assert np.array_equal(getattr(net, q).value, getattr(net_chk, q).value), q
assert [type(c) for c in net.costs] == [type(c) for c in net_chk.costs]
print('[ok] make_network == unrolled build')

## 5) Frame `K` — `evaluation.py::experiment_tracking` (per-frame inputs)
Below one frame `K` (default 0) is followed iteration by iteration. For `K > 0` the network first processes frames
`0..K−1` exactly as the harness does, so frame `K` starts from the carried-over (warm-started) maps.
The gyro over the window is this frame's anchor target (`step(V, omega_imu=ω_gyro)`). The reference `ω_ref` (Vicon, differenced) is used only to score.

In [ ]:
# evaluation.py::experiment_tracking — frames before K (warm start), then frame K's inputs
for k_prev, (V_k, _) in enumerate(frames[:K]):
    omega_imu_k = get_gyro_for_frame(imu_data, rc.t_start + k_prev * rc.frame_duration,
                                     rc.t_start + (k_prev + 1) * rc.frame_duration)
    net.step(V_k, n_iters=rc.n_iters, omega_imu=omega_imu_k)
t_lo = rc.t_start + K * rc.frame_duration
t_hi = t_lo + rc.frame_duration
V, t_mid = frames[K]
omega_imu = get_gyro_for_frame(imu_data, t_lo, t_hi)
omega_ref, ref_src = get_reference_omega(gt_data, imu_data, t_lo, t_hi, om_data)
print(f'frame {K}: [{t_lo:.4f}, {t_hi:.4f})  t_mid {t_mid:.4f}')
print('ω gyro  °/s :', to_deg_s(omega_imu))
print(f'ω ref   °/s : {to_deg_s(omega_ref)}   ({ref_src})')
print('network R at frame start °/s:', to_deg_s(net.R / rc.frame_duration))

## 6) One iteration — `InteractingMapsThesis.step` body, unrolled (`n_iters = 1`)
`step` stores `V`, sets the IMU switch, then per iteration: **Phase 1** reset the accumulators, every active cost reads
the *same* `{I,G,F,R}` and adds its message; **Phase 2** every quantity does `value −= accumulator`, then clip.
Below, each cost's message is computed by hand into `msgs[cost][quantity]`; a check then calls the real
`compute_and_send_gradients` on the same state and compares. `net0` keeps the frame-start state.

In [ ]:
# keep the frame-start state for the checks
net0 = copy.deepcopy(net)

In [ ]:
# network_dissertation.py::step — inputs and the IMU switch, then Phase 1 reset
net.q_V.value = V
net.cost_imu.R_imu = omega_imu * net.frame_duration
net._use_imu = True
for q in [net.q_I, net.q_G, net.q_F, net.q_R]:
    q.reset_gradient()
msgs = {}
print('active costs:', [type(c).__name__ for c in net.costs if not (c is net.cost_imu and not net._use_imu)])

### 6a) `Cost_OFCE` — `V + F·G = 0`
`e = V + F·G`; raw gradients `∂/∂F = 2e·G`, `∂/∂G = 2e·F`; each component clipped to `±max_grad = 5`, then × `δ_VFG`.
Jacobi: both read the *old* F and G (contrast Cook, where G's step sees the new F). On the first iteration `G = 0`, so
the F message is zero and the G message is `2δ_VFG·V·F` (a descent message: G moves to `−2δ_VFG·V·F`, parallel to the
flow, sign set by the events).

In [ ]:
# network_dissertation.py::Cost_OFCE.compute_and_send_gradients
c_ofce = net.costs[0]
error = net.q_V.value + np.sum(net.F * net.G, axis=-1)
grad_F = 2.0 * error[..., np.newaxis] * net.G
grad_G = 2.0 * error[..., np.newaxis] * net.F
assert not c_ofce.normalize                     # reported numbers use the raw step
clip_share = (np.mean(np.abs(grad_F) > c_ofce.max_grad), np.mean(np.abs(grad_G) > c_ofce.max_grad))
grad_F = np.clip(grad_F, -c_ofce.max_grad, c_ofce.max_grad)
grad_G = np.clip(grad_G, -c_ofce.max_grad, c_ofce.max_grad)
msgs['Cost_OFCE'] = {'F': grad_F * c_ofce.delta_VFG, 'G': grad_G * c_ofce.delta_VFG}
print(f'|e| mean {np.abs(error).mean():.4f} | share clipped at max_grad: F {clip_share[0]:.3%}, G {clip_share[1]:.3%}')
print('|msg F| max', np.abs(msgs['Cost_OFCE']['F']).max(), '| |msg G| max', np.abs(msgs['Cost_OFCE']['G']).max())

### 6b) `Cost_Spatial` — `G = ∇I`
`∇I` by forward differences on the `H×W` grid, with the last column of `∂I/∂x` and last row of `∂I/∂y` set to 0
(Neumann border; Cook instead uses an extra row/column of I). `error = G − ∇I`. G message `δ_IG·error` (a blend toward
∇I). I message, `poisson='iterative'` (Eq. 6.61): the backward-difference divergence of `error`, × `δ_GI` — one
Richardson step on `‖∇I − G‖²`. (`'fft'`/`'dct'` instead send `I − I*` from an exact solve; this walkthrough covers
the default.)

In [ ]:
# network_dissertation.py::Cost_Spatial.compute_and_send_gradients — ∇I with zero last column/row
c_sp = net.costs[1]
i_map = net.I
grad_I_x = np.zeros_like(i_map); grad_I_x[:, :-1] = i_map[:, 1:] - i_map[:, :-1]
grad_I_y = np.zeros_like(i_map); grad_I_y[:-1, :] = i_map[1:, :] - i_map[:-1, :]
err_sp = net.G - np.stack([grad_I_x, grad_I_y], axis=-1)
print('poisson:', c_sp.poisson, '| shrink:', c_sp.shrink, '| |G − ∇I| mean', np.abs(err_sp).mean())

In [ ]:
# network_dissertation.py::Cost_Spatial — G message and the iterative (Eq. 6.61) I message
assert c_sp.poisson == 'iterative' and c_sp.shrink == 0.0
grad_I_update = err_sp[:, :, 0] + err_sp[:, :, 1]
grad_I_update[:, 1:] -= err_sp[:, :-1, 0]
grad_I_update[1:, :] -= err_sp[:-1, :, 1]
msgs['Cost_Spatial'] = {'G': err_sp * c_sp.delta_IG, 'I': grad_I_update * c_sp.delta_GI}
print('|msg G| max', np.abs(msgs['Cost_Spatial']['G']).max(), '| |msg I| max', np.abs(msgs['Cost_Spatial']['I']).max())

### 6c) `Cost_Kinematics` — `F = C·R`
F message `δ_RF·(F − C·R)` (blend toward the rigid-rotation flow). R message `δ_FR·(R − R*)` with
`R* = M⁻¹ΣCᵀF` — a gradient step toward the least-squares R of the current flow. On frame 0 `F = C·R` exactly, so both
messages start at zero.

In [ ]:
# network_dissertation.py::Cost_Kinematics.compute_and_send_gradients
c_kin = net.cost_kin
f_target = np.einsum('hwij,j->hwi', c_kin.C_mat, net.R)
R_target = solve_R_lstsq(c_kin._M_inv, c_kin.C_mat, net.F)
msgs['Cost_Kinematics'] = {'F': (net.F - f_target) * c_kin.delta_RF, 'R': (net.R - R_target) * c_kin.delta_FR}
print('R* °/s:', to_deg_s(R_target / rc.frame_duration), '| R msg °/s-equiv:', to_deg_s(msgs['Cost_Kinematics']['R'] / rc.frame_duration))

### 6d) `Cost_IMU` — the anchor message
`R msg = δ_IMU·(R − ω_gyro·dt)`. On frame 0 `R = initial_R` *is* the gyro over the same window, so this message is
exactly zero on the first iteration; it only acts once R* has moved R away from the gyro (or the gyro changes).

In [ ]:
# network_dissertation.py::Cost_IMU.compute_and_send_gradients
msgs['Cost_IMU'] = {'R': (net.R - net.cost_imu.R_imu) * net.cost_imu.delta_IMU}
print('anchor R_imu °/s:', to_deg_s(net.cost_imu.R_imu / rc.frame_duration),
      '| R msg °/s-equiv:', to_deg_s(msgs['Cost_IMU']['R'] / rc.frame_duration))

In [ ]:
# check: every cost's real compute_and_send_gradients == the hand-written messages
for cost in net.costs:
    name = type(cost).__name__
    if name not in msgs:
        continue
    chk = copy.deepcopy(net)                       # accumulators are still all zero here
    chk.costs[net.costs.index(cost)].compute_and_send_gradients()
    for qn, arr in msgs[name].items():
        assert np.array_equal(getattr(chk, 'q_' + qn).gradient_accumulator, arr), (name, qn)
    others = {'I', 'G', 'F', 'R'} - set(msgs[name])
    assert all(not np.any(getattr(chk, 'q_' + o).gradient_accumulator) for o in others), name
    print(f'[ok] {name}: messages on {sorted(msgs[name])}')

### 6e) Accumulate, then Phase 2 — `Quantity.add_gradient`, `Quantity.update(1.0)`, clip
Each accumulator is the sum of the messages it received; every quantity moves at once by `−accumulator`. Then
I, G, F, R are clipped to ±10, ±5, ±10, ±1 (R in rad/frame — ±50 rad/s at 20 ms, never hit).

In [ ]:
# network_dissertation.py::step — Phase 1 accumulation (in cost order) and Phase 2 update + clip
for name, m in msgs.items():
    for qn, arr in m.items():
        getattr(net, 'q_' + qn).add_gradient(arr)
for q in [net.q_I, net.q_G, net.q_F, net.q_R]:
    q.update(1.0)
net.q_I.value = np.clip(net.q_I.value, -CLIP_I, CLIP_I)
net.q_G.value = np.clip(net.q_G.value, -CLIP_G, CLIP_G)
net.q_F.value = np.clip(net.q_F.value, -CLIP_F, CLIP_F)
net.q_R.value = np.clip(net.q_R.value, -CLIP_R, CLIP_R)
print('R after 1 iteration °/s:', to_deg_s(net.R / rc.frame_duration))
print('share at bound: F', np.mean(np.abs(net.F) >= CLIP_F), 'G', np.mean(np.abs(net.G) >= CLIP_G))

In [ ]:
# check: InteractingMapsThesis.step(V, n_iters=1) == the unrolled iteration
net_ref = copy.deepcopy(net0)
net_ref.step(V, n_iters=1, omega_imu=omega_imu)
for q in ('q_I', 'q_G', 'q_F', 'q_R'):
    assert np.allclose(getattr(net_ref, q).value, getattr(net, q).value, rtol=0, atol=1e-15), q
print('[ok] step(n_iters=1) == unrolled iteration')

## 7) The `n_iters` loop on frame `K` — `InteractingMapsThesis.step` (unrolled)
Restart from `net0` and run all iterations exactly as `step` does, recording each cost's message on R, the
flow-implied `R*`, and the constraint residuals. Two messages compete on R: `δ_FR·(R − R*)` and `δ_IMU·(R − ω_gyro·dt)`; with R* frozen, R would settle at the weighted mean `(δ_FR·R* + δ_IMU·R_imu)/(δ_FR+δ_IMU)`.

In [ ]:
# network_dissertation.py::step — the n_iters loop, recording each cost's message on R
net = copy.deepcopy(net0)
net.q_V.value = V
net.cost_imu.R_imu = omega_imu * net.frame_duration
net._use_imu = True
R_iters, R_msgs, R_star, res_vfg, res_gi = [net.q_R.value.copy()], [], [], [], []
res_fcr, R_star_post = [], []   # kinematic residual and R* after each iteration (for the CSV)
for _ in range(rc.n_iters):
    # PHASE 1
    for q in [net.q_I, net.q_G, net.q_F, net.q_R]:
        q.reset_gradient()
    msgs_it = {}
    for cost in net.costs:
        if cost is net.cost_imu and not net._use_imu:
            continue
        before = net.q_R.gradient_accumulator.copy()
        cost.compute_and_send_gradients()
        msgs_it[type(cost).__name__] = net.q_R.gradient_accumulator - before
    R_msgs.append(msgs_it)
    R_star.append(net.R - msgs_it['Cost_Kinematics'] / net.cost_kin.delta_FR)   # implied R* this iter
    # PHASE 2
    net.q_I.update(1.0); net.q_G.update(1.0); net.q_F.update(1.0); net.q_R.update(1.0)
    net.q_I.value = np.clip(net.q_I.value, -CLIP_I, CLIP_I)
    net.q_G.value = np.clip(net.q_G.value, -CLIP_G, CLIP_G)
    net.q_F.value = np.clip(net.q_F.value, -CLIP_F, CLIP_F)
    net.q_R.value = np.clip(net.q_R.value, -CLIP_R, CLIP_R)
    R_iters.append(net.q_R.value.copy()); res_vfg.append(net.residual_VFG(V)); res_gi.append(net.residual_GI())
    res_fcr.append(net.residual_FCR()); R_star_post.append(solve_R_lstsq(net.cost_kin._M_inv, net._C_mat, net.F))
R_iters, R_star = np.array(R_iters), np.array(R_star)
print('R after frame °/s:', to_deg_s(net.R / rc.frame_duration))

In [ ]:
# check: InteractingMapsThesis.step == unrolled loop
net_ref = copy.deepcopy(net0)
net_ref.step(V, n_iters=rc.n_iters, omega_imu=omega_imu)
for q in ('q_I', 'q_G', 'q_F', 'q_R'):
    assert np.array_equal(getattr(net_ref, q).value, getattr(net, q).value), q
print('[ok] step() == unrolled loop')

In [ ]:
# who moved R: message sizes (°/s-equivalent) at the first/last iteration
for i in (0, rc.n_iters - 1):
    print(f'iter {i:3d}', {k: to_deg_s(v / rc.frame_duration).round(3) for k, v in R_msgs[i].items() if np.any(v)})
print('flow-implied R* at the end °/s:', to_deg_s(R_star[-1] / rc.frame_duration))
wm = (net.cost_kin.delta_FR * R_star[-1] + net.cost_imu.delta_IMU * net.cost_imu.R_imu) / \
     (net.cost_kin.delta_FR + net.cost_imu.delta_IMU)
print('weighted-mean prediction °/s:', to_deg_s(wm / rc.frame_duration), '| actual', to_deg_s(net.R / rc.frame_duration))

In [ ]:
# R over the iterations vs the flow-implied R*, the reference and the gyro (°/s)
it = np.arange(rc.n_iters + 1)
with use_style('screen'):
    fig, axes = plt.subplots(1, 3, figsize=(SCREEN_WIDTH, 3), sharex=True)
    for a, ax in enumerate(axes):
        ax.plot(it, to_deg_s(R_iters[:, a] / rc.frame_duration), color=OMEGA_AXIS_COLORS[a], label='network R')
        ax.plot(it[:-1], to_deg_s(R_star[:, a] / rc.frame_duration), color=OMEGA_AXIS_COLORS[a], ls='--', lw=0.8, label='R* (flow)')
        ax.axhline(to_deg_s(omega_ref[a]), color='k', lw=1.2, label='reference')
        ax.axhline(to_deg_s(omega_imu[a]), color='m', lw=0.9, label='anchor ω_gyro')
        ax.set_title(f'$\\omega_{"xyz"[a]}$ (°/s)'); ax.set_xlabel('iteration')
    axes[0].legend(fontsize=7)
    path = os.path.join(FIG_DIR, f'{MODEL}_R_over_iterations.png'); save(fig, path)
display(Image(path))

In [ ]:
# constraint residuals over the iterations: mean |V + F·G| and mean |G − ∇I|
with use_style('screen'):
    fig, axes = plt.subplots(1, 2, figsize=(SCREEN_WIDTH * 2 / 3, 2.6), sharex=True)
    axes[0].plot(it[1:], res_vfg, color='0.25'); axes[0].set_title(r'mean $|V + F\cdot G|$')
    axes[1].plot(it[1:], res_gi, color='0.25'); axes[1].set_title(r'mean $|G - \nabla I|$')
    for ax in axes:
        ax.set_xlabel('iteration')
    path = os.path.join(FIG_DIR, f'{MODEL}_residuals.png'); save(fig, path)
display(Image(path))

In [ ]:
# save every residual per iteration of frame K (state after each iteration), for combined figures later
# R, R*, ω in °/s; res_RRstar = |R − R*| (norm, °/s); res_IMU = |R − ω_gyro·dt| (norm, °/s), the anchor
deg = lambda r: to_deg_s(np.asarray(r) / rc.frame_duration)
it_df = pd.DataFrame({'iteration': it[1:],
                      **{f'R_w{c}': deg(R_iters[1:, a]) for a, c in enumerate('xyz')},
                      **{f'Rstar_w{c}': deg(np.array(R_star_post)[:, a]) for a, c in enumerate('xyz')},
                      'res_VFG': res_vfg, 'res_GI': res_gi, 'res_FCR': res_fcr,
                      'res_RRstar': np.linalg.norm(deg(R_iters[1:]) - deg(R_star_post), axis=1),
                      'res_IMU': np.linalg.norm(deg(R_iters[1:]) - to_deg_s(omega_imu), axis=1),
                      })
csv_path = os.path.join(FIG_DIR, f'{MODEL}_iterations.csv'); it_df.to_csv(csv_path, index=False)
print(csv_path, it_df.shape); it_df.iloc[[0, 1, -1]].round(4)

## 8) The maps after frame `K` — what the network inferred
`V` and the three inferred maps (cropped to `H×W` by `viz.net_maps`). Check whether `G` is concentrated on the event
pixels, whether `F` still looks like a rigid rotation field (`C·R`) or has picked up local structure from OFCE, and how
much of G is not any image's gradient (`metrics.curl_share` — the part the I recovery must discard).

In [ ]:
# viz.net_maps + draw_map — V, I, G, F after frame K
maps = net_maps(net, H, W)
with use_style('screen'):
    fig, axes = plt.subplots(1, 4, figsize=(SCREEN_WIDTH, 2.6))
    draw_map(axes[0], 'V', V, title=f'V (frame {K})')
    draw_map(axes[1], 'I', maps['I'])
    draw_map(axes[2], 'G', maps['G'])
    draw_map(axes[3], 'F', maps['F'])
    path = os.path.join(FIG_DIR, f'{MODEL}_maps_frame{K}.png'); save(fig, path)
display(Image(path))

In [ ]:
# numbers behind the picture: residuals, curl share, clip hits, F vs the rigid field C·R
F_rigid = np.einsum('hwij,j->hwi', C_mat, net.R)
ev = V != 0
print(f'mean |V+F·G| {net.residual_VFG(V):.4f} | mean |G−∇I| {net.residual_GI():.4f} | curl share of G {curl_share(maps["G"]):.2%}')
print(f'|G| mean on event px {np.hypot(*maps["G"][ev].T).mean():.4f} vs off {np.hypot(*maps["G"][~ev].T).mean():.4f}')
print(f'|F − C·R| / |C·R| = {np.linalg.norm(maps["F"] - F_rigid) / np.linalg.norm(F_rigid):.2%}')
print('share at clip bound: F', np.mean(np.abs(maps['F']) >= CLIP_F), 'G', np.mean(np.abs(maps['G']) >= CLIP_G),
      'I', np.mean(np.abs(maps['I']) >= CLIP_I), '| I std', maps['I'].std())
e, d, b = compute_metrics(net.R / rc.frame_duration, omega_ref)
print(f'frame {K}: err {e:.2f} °/s, dir {d:.2f}°, β {b:.3f}')

## 9) All frames — `evaluation.py::experiment_tracking` (`thesis_imu` branch, unrolled)
Fresh network, then per frame: gyro + reference for the window → `rc.use_thesis and rc.use_imu`: `net.step(V, omega_imu=ω_gyro)` → `ω_est = R/dt` → `compute_metrics`
against the reference. The maps are carried from frame to frame (warm start); `R` is never reset.

In [ ]:
# evaluation.py::experiment_tracking — the frame loop
net = make_network(rc, H, W, fx, fy, cx, cy)
rows = []
for k, (V_k, t_mid_k) in enumerate(frames):
    t_lo_k = rc.t_start + k * rc.frame_duration
    t_hi_k = t_lo_k + rc.frame_duration
    omega_imu_k = get_gyro_for_frame(imu_data, t_lo_k, t_hi_k)
    omega_ref_k, _ = get_reference_omega(gt_data, imu_data, t_lo_k, t_hi_k, om_data)
    net.step(V_k, n_iters=rc.n_iters, omega_imu=omega_imu_k)
    omega_est = net.R / rc.frame_duration
    err, dir_err, beta = compute_metrics(omega_est, omega_ref_k)
    rows.append({'frame': k, 't_mid': t_mid_k,
                 **{f'est_w{c}': v for c, v in zip('xyz', to_deg_s(omega_est))},
                 **{f'ref_w{c}': v for c, v in zip('xyz', to_deg_s(omega_ref_k))},
                 **{f'imu_w{c}': v for c, v in zip('xyz', to_deg_s(omega_imu_k))},
                 'err_est': err, 'err_gyro': compute_metrics(omega_imu_k, omega_ref_k)[0],
                 'dir_err': dir_err, 'beta': beta,
                 'res_VFG': net.residual_VFG(V_k), 'res_GI': net.residual_GI(), 'res_FCR': net.residual_FCR(),
                 'I_std': float(net.I[:H, :W].std()), 'curl_share': curl_share(net.G[:H, :W])})
df = pd.DataFrame(rows)
df.round(3)

In [ ]:
# summary (°/s): network vs the gyro, both against the reference
df[['err_est', 'err_gyro', 'dir_err', 'beta', 'res_VFG', 'res_GI', 'res_FCR', 'curl_share']].describe().round(3)

In [ ]:
# save the per-frame table (ω in °/s, residuals after each frame), for combined figures later
csv_path = os.path.join(FIG_DIR, f'{MODEL}_frames.csv'); df.to_csv(csv_path, index=False)
print(csv_path, df.shape)

In [ ]:
# check: evaluation.experiment_tracking (thesis_imu) == unrolled loop  (writes to experiments/model_walkthrough/)
E.experiment_tracking(rc, save_frames=False)
trk = pd.read_csv(os.path.join(rc.output_dir, 'tracking.csv'))
assert np.allclose(np.degrees(trk[['est_wx', 'est_wy', 'est_wz']].values),
                   df[['est_wx', 'est_wy', 'est_wz']].values, atol=1e-9)
assert np.allclose(trk['err_deg_s'].values, df['err_est'].values, atol=1e-9)
assert np.allclose(trk['I_std'].values, df['I_std'].values, atol=1e-12)
print('[ok] experiment_tracking == unrolled thesis_imu loop')

In [ ]:
# ω per axis over the frames: estimate vs reference vs gyro (°/s)
t_axis = df['t_mid'] - rc.t_start
with use_style('screen'):
    fig, axes = plt.subplots(1, 3, figsize=(SCREEN_WIDTH, 3.2), sharex=True)
    for a, ax in enumerate(axes):
        c = 'xyz'[a]
        ax.plot(t_axis, df[f'est_w{c}'], color=OMEGA_AXIS_COLORS[a], label='network ω')
        ax.plot(t_axis, df[f'ref_w{c}'], color='k', lw=1.2, label='reference')
        ax.plot(t_axis, df[f'imu_w{c}'], color='m', ls=':', label='gyro (anchor)')
        ax.set_title(f'$\\omega_{c}$ (°/s)')
        ax.set_xlabel('time (s)')
    axes[0].legend(fontsize=7)
    path = os.path.join(FIG_DIR, f'{MODEL}_omega_over_frames.png')
    save(fig, path)
display(Image(path))

## What to check
- **6d frame 0:** the anchor equals the start R (both are the same gyro window), so `Cost_IMU` contributes nothing
  until R* pulls R away — the IMU acts as a restoring force, not an initial push.
- **7 weighted mean:** with `δ_FR = 0.10`, `δ_IMU = 0.50` the final R is ~5:1 gyro:flow. Does the weighted-mean
  prediction match the actual R? How far is R* (flow) from the gyro and from the reference?
- **9 `err_est` vs `err_gyro`:** both scored against Vicon. If `err_est > err_gyro`, the vision part only adds noise on
  top of the gyro; if smaller, OFCE corrects gyro bias/latency.
- **6a OFCE clip / 8 clip hits:** a strong anchor keeps |F| at the true flow, which can be larger than the
  vision-only network lets it get — check whether the OFCE step clip and the G value clip are hit more often than in
  the `thesis` notebook.
- **8 curl share and I:** the anchor fixes R, not G; compare curl share and I std with the `thesis` notebook.